# Data Collection & Initial Processing Overview

This notebook provides a step-by-step summary of the data collection phase, data storage setup, and initial dataset processing pipeline.

--- 

## 1. Overview of Data Collection Phases

| Phase / Source | Target Content | Collection Script Path | Output Directory |
|---|---|---|---|
| **GovInfo API** | U.S. Congressional Hearing Package Manifests, MODS metadata, and HTML transcripts (1995–2025) | [`data_collection/govinfo/collect_govinfo.py`](../data_collection/govinfo/collect_govinfo.py) | `data_files/govinfo/` |
| **C-SPAN Archives API** | Editorial Abstracts for House, Senate, and Joint Committee Programs | [`data_collection/cspan/collect_cspan.py`](../data_collection/cspan/collect_cspan.py) | `data_files/cspan/` |
| **SEC-API.io** | Bulk Form 10-K ZIP containers (1993–present) & Extracted Items (1, 1A, 7) | [`data_collection/sec_api/collect_sec.py`](../data_collection/sec_api/collect_sec.py) | `data_files/sec_api/` |

--- 

## 2. Detailed Data Collection Workflows

### Phase A: GovInfo Congressional Hearing Collection
- **Source**: Official U.S. Government Publishing Office (GPO) GovInfo API (`CHRG` collection).
- **Methodology**:
  1. **Enumeration**: Year-by-year cursor walk using `offsetMark` pagination to avoid shifting offset drops.
  2. **Retrieval**: Download MODS XML (`/packages/{id}/mods`) and raw text HTML transcripts (`/packages/{id}/htm`).
- **Script**: Executable pipeline template stored in `data_collection/govinfo/collect_govinfo.py`.

### Phase B: C-SPAN Reference Abstract Collection
- **Source**: C-SPAN Archives API (`api.c-spanarchives.org`).
- **Methodology**:
  - Queries Lucene formatted ISO dates across three specific committee categories (`House Committee`, `Senate Committee`, `Joint Committee`).
  - Collects independent, human-written editorial summaries (~48 words median) to evaluate generated summaries.
- **Script**: Executable pipeline template stored in `data_collection/cspan/collect_cspan.py`.

### Phase C: SEC Form 10-K Corporate Filings
- **Source**: SEC-API.io Bulk & Extractor APIs.
- **Methodology**:
  - Downloads 394 monthly bulk ZIP archives (~34 GB total) containing ~305,140 annual reports.
  - Extracts specific item sections (`Item 1: Business`, `Item 1A: Risk Factors`, `Item 7: MD&A`).
- **Script**: Executable pipeline template stored in `data_collection/sec_api/collect_sec.py`.

--- 

## 3. Initial Data Processing & Structuring

The raw datasets collected in `data_files/` are processed by [`data_processing/initial_processing.py`](../data_processing/initial_processing.py):
1. **MODS XML Parsing**: Extracts structured speaker rosters, witness lists, and committee affiliations.
2. **Transcript Turn Attribution**: Parses HTML transcripts using the 4-space paragraph indent rule to extract speaker turns.
3. **Corpus Assembly**: Merges transcripts with metadata and C-SPAN reference summaries into standardized Parquet tables for downstream LLM analysis.